# Environment Setup

Run this notebook before any other notebook in the tutorial. It checks your
environment, installs `oneccl_bindings_for_pytorch` if needed, inspects your
hardware topology, and runs a small multi-rank allreduce to confirm everything works end-to-end.

---

## How These Notebooks Work

All notebooks in this tutorial use a **launcher pattern**:

- You run each notebook from a **normal single-rank Jupyter kernel** — no `mpirun` needed to start Jupyter
- Cells that benchmark multi-rank collectives write a Python script to `/tmp/` and execute it with `mpirun` as a subprocess
- Output from all ranks appears in the cell below the launcher

```
%%writefile /tmp/my_script.py    ← writes the multi-rank script to disk
import torch.distributed as dist
...

result = subprocess.run("mpirun -n 4 python /tmp/my_script.py", ...)
print(result.stdout)             ← all 4 ranks' output appears here
```

**You do not need to launch Jupyter with mpirun.**

### Starting Jupyter on a BMG/CRI node

```bash
# 1. SSH into your BMG/CRI node (or open a JupyterHub terminal)

# 2. Load the Intel oneAPI environment — this sets up mpirun, CCL_ROOT, etc.
source /opt/intel/oneapi/setvars.sh
# or, if your cluster uses modules:
module load intel/mpi intel/oneapi/pytorch

# 3. Launch JupyterLab (single kernel — no mpirun prefix needed)
jupyter lab --no-browser --port=8888
# Open the printed URL in your browser, or use the VS Code Jupyter extension
```

If `mpirun` or `oneccl_bindings_for_pytorch` are missing after step 2, the environment
check cell below will tell you exactly what is wrong.

## Step 1 — Check Environment

In [ ]:
import subprocess, sys, shutil

def check(label, cmd, expect_in=None, hint=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = result.stdout + result.stderr
    ok = result.returncode == 0 and (expect_in is None or expect_in in out)
    status = "OK  " if ok else "FAIL"
    print(f"[{status}] {label}")
    if not ok:
        print(f"       got: {out.strip()[:200] or '(no output)'}")
        if hint:
            print(f"       fix: {hint}")
    return ok

print("=== Runtime ===")
check("Python 3.9+",
      f"{sys.executable} --version",
      hint="Use a Python 3.9+ environment")

check("PyTorch installed",
      f"{sys.executable} -c 'import torch; print(torch.__version__)'",
      hint="pip install torch (Intel XPU build) or load the oneapi/pytorch module")

check("XPU device available",
      f"{sys.executable} -c 'import torch; assert torch.xpu.is_available(), \"no XPU\"'",
      hint="Run this notebook on a node with an Intel GPU and the XPU-enabled PyTorch build")

print()
print("=== MPI / CCL ===")
check("mpirun in PATH",
      "which mpirun",
      hint="Run: source /opt/intel/oneapi/setvars.sh  (or: module load intel/mpi)")

check("Intel MPI (not OpenMPI)",
      "mpirun --version",
      expect_in="Intel",
      hint="oneCCL requires Intel MPI. OpenMPI is not supported. Load intel/mpi module.")

check("CCL_ROOT set",
      "test -n \"$CCL_ROOT\"",
      hint="Run: source /opt/intel/oneapi/setvars.sh to set CCL_ROOT")

check("oneccl_bindings_for_pytorch importable",
      f"{sys.executable} -c 'import oneccl_bindings_for_pytorch'",
      hint="See Step 2 below to install or locate the package")

print()
print("=== Tools ===")
check("numactl available",
      "which numactl",
      hint="Install: sudo apt install numactl  (used for NUMA topology inspection)")

## Step 2 — Install oneCCL Bindings (if needed)

**On a BMG/CRI cluster or managed JupyterHub:** `oneccl_bindings_for_pytorch` is almost
certainly already installed as part of the `intel/oneapi/pytorch` module. If Step 1
shows `[OK  ] oneccl_bindings_for_pytorch importable`, skip this step entirely.

**On a bare machine or custom environment:** install with pip. The package name on Intel's
PyPI channel is `oneccl_bind_pt`. Pin the version to match your PyTorch build:

```bash
# Find your PyTorch version first:
python -c "import torch; print(torch.__version__)"

# Install the matching oneccl bindings:
pip install oneccl_bind_pt==<your_torch_version> \
    --extra-index-url https://pytorch-extension.intel.com/release-whl/stable/xpu/us/
```

The cell below checks whether installation is needed before doing anything:

In [ ]:
import subprocess, sys

# Check if already importable — skip install if so
result = subprocess.run(
    f"{sys.executable} -c 'import oneccl_bindings_for_pytorch; "
    "import oneccl_bindings_for_pytorch as b; print(b.__version__)'",
    shell=True, capture_output=True, text=True
)

if result.returncode == 0:
    print(f"oneccl_bindings_for_pytorch already installed: {result.stdout.strip()}")
    print("Nothing to do — proceed to Step 3.")
else:
    print("Not found. Installing from Intel PyPI...")
    import torch
    torch_ver = torch.__version__.split("+")[0]  # strip local tag e.g. "+xpu"
    install_cmd = (
        f"{sys.executable} -m pip install oneccl_bind_pt=={torch_ver} "
        "--extra-index-url https://pytorch-extension.intel.com/release-whl/stable/xpu/us/"
    )
    print(f"Running: {install_cmd}\n")
    r = subprocess.run(install_cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print("INSTALL FAILED:")
        print(r.stderr[-1000:])
        print()
        print("Manual fix options:")
        print(f"  1. pip install oneccl_bind_pt=={torch_ver} "
              "--extra-index-url https://pytorch-extension.intel.com/release-whl/stable/xpu/us/")
        print("  2. On BMG/CRI: module load intel/oneapi/pytorch  (includes the bindings)")
    else:
        # Verify the install actually worked
        v = subprocess.run(
            f"{sys.executable} -c 'import oneccl_bindings_for_pytorch as b; print(b.__version__)'",
            shell=True, capture_output=True, text=True
        )
        if v.returncode == 0:
            print(f"\nInstalled successfully: {v.stdout.strip()}")
        else:
            print("\nInstall appeared to succeed but import still fails:")
            print(v.stderr[:500])
            print("Check for PyTorch version mismatch (oneccl_bind_pt must match torch version exactly)")

## Step 3 — Hardware Topology

Check your GPU count and NUMA layout before running any collective benchmark.
**This matters**: oneCCL constructs topology-aware rings at init time. If GPU-to-NUMA
mapping is wrong or unexpected, ring performance will be 2–5× worse.

Expected healthy output for a 2-socket BMG/CRI node with 4 GPUs (2 per socket):

```
GPU0 → NUMA node 0    GPU1 → NUMA node 0
GPU2 → NUMA node 1    GPU3 → NUMA node 1
node distances: 0→0: 10,  0→1: 21
```

If all GPUs show the same NUMA node, or NUMA distances are all 10, the topology
detection may be wrong — check `dmesg | grep -i iommu` and `/sys/bus/pci/.../numa_node`.

In [ ]:
import subprocess, sys

# XPU device list
print("=== XPU Devices ===")
r = subprocess.run(
    f"{sys.executable} -c \""
    "import torch; n = torch.xpu.device_count(); "
    "print(f'{n} XPU device(s) found'); "
    "[print(f'  xpu:{i}  {torch.xpu.get_device_properties(i).name}') for i in range(n)]\"",
    shell=True, capture_output=True, text=True
)
print(r.stdout or r.stderr)

# NUMA topology
print("=== NUMA Topology ===")
r2 = subprocess.run("numactl --hardware 2>&1", shell=True, capture_output=True, text=True)
if r2.returncode == 0:
    for line in r2.stdout.splitlines():
        if any(k in line for k in ("available", "node", "distance", "size")):
            print(line)
else:
    print("numactl not found — install with: sudo apt install numactl")

# GPU-to-NUMA affinity
print()
print("=== GPU → NUMA Mapping ===")
r3 = subprocess.run(
    "for i in $(seq 0 7); do "
    "  dev=$(ls /dev/dri/renderD$((128+i)) 2>/dev/null) || continue; "
    "  pci=$(udevadm info --query=path --name=$dev 2>/dev/null | grep -o 'pci[^/]*' | tail -1) || continue; "
    "  numa=$(cat /sys/bus/pci/devices/$pci/numa_node 2>/dev/null || echo '?'); "
    "  echo \"GPU$i -> NUMA node $numa\"; "
    "done",
    shell=True, capture_output=True, text=True
)
if r3.stdout.strip():
    print(r3.stdout)
else:
    # Fallback using Level Zero device properties if udevadm path fails
    print("(udevadm path not available, try: ls /sys/class/drm/renderD*/device/numa_node)")

## Step 4 — End-to-End Verification

Runs a 4-rank allreduce via `mpirun`. Each rank contributes `rank+1`; the correct
sum is `1+2+3+4 = 10`. All four ranks must print `PASS`.

If this fails, fix it before proceeding — every other notebook depends on this working.

In [ ]:
%%writefile /tmp/ccl_verify.py
import os, sys
import torch
import torch.distributed as dist
import oneccl_bindings_for_pytorch

os.environ["CCL_ATL_TRANSPORT"] = "ofi"
os.environ["CCL_WORKER_COUNT"]  = "1"
os.environ["CCL_LOG_LEVEL"]     = "warn"

dist.init_process_group(backend="ccl")

rank   = dist.get_rank()
size   = dist.get_world_size()
device = torch.device(f"xpu:{rank % torch.xpu.device_count()}")

# Each rank contributes (rank + 1); sum should be size*(size+1)/2
t = torch.tensor([float(rank + 1)], device=device)
dist.all_reduce(t, op=dist.ReduceOp.SUM)
torch.xpu.synchronize(device)

expected = size * (size + 1) / 2
status   = "PASS" if abs(t.item() - expected) < 0.01 else "FAIL"
print(f"[rank {rank}/{size}]  device={device}  result={t.item():.0f}  expected={expected:.0f}  {status}")

dist.destroy_process_group()

In [ ]:
import subprocess

n_gpus = __import__("torch").xpu.device_count()
n_ranks = min(n_gpus, 4)  # use however many GPUs are available, up to 4

result = subprocess.run(
    f"mpirun -n {n_ranks} -ppn {n_ranks} python /tmp/ccl_verify.py",
    shell=True, capture_output=True, text=True
)
print(result.stdout)

if result.returncode != 0:
    print("FAILED. stderr:")
    print(result.stderr[-2000:])
    print()
    print("Common causes:")
    print("  mpirun not found          → source /opt/intel/oneapi/setvars.sh")
    print("  import oneccl_bindings... → complete Step 2 above")
    print("  no XPU device             → must run on a node with Intel GPU")
    print("  CCL_ATL_TRANSPORT error   → try: export CCL_ATL_TRANSPORT=mpi")
elif "FAIL" in result.stdout:
    print("Allreduce returned wrong values — check for driver or version mismatch")
else:
    print(f"All {n_ranks} ranks passed. Environment is ready.")

---

All steps passed? Proceed to **[Allreduce — The TP Decode Hot Path](03a_allreduce_walkthrough.ipynb)**.

The notebooks run in order: `03a` (Allreduce) → `03b` (Allgather) → `03c` (Alltoall/MoE) → `04` (End-to-End TP Decode).
Each one uses the same launcher pattern you just saw above.